# MIMII Anomalib DataModule

This notebook uses a custom Lightning DataModule that reads `data/dcase-2020-spectrogram/meta.json` generated by your `mimii-toy.py` script.

In [1]:
import importlib
import sys
from pathlib import Path

from anomalib.engine import Engine

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.mimii_anomalib_datamodule as mimii_dm_module
importlib.reload(mimii_dm_module)
MIMIIAnomalibDataModule = mimii_dm_module.MIMIIAnomalibDataModule

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from torchvision.transforms.v2 import Compose, Resize, Transform, ToTensor
dm = MIMIIAnomalibDataModule(
    root="../data/dcase-2020-spectrogram",
    categories=("fan", "pump", "slider", "valve"),
    train_phases=("id_00",),
    val_phases=("id_04",),
    test_phases=("id_02",),
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=8,
    augmentations=Compose([
        Resize((256, 256)),
        ToTensor()]
    ),
)

dm.setup("fit")
train_batch = next(iter(dm.train_dataloader()))
print("train keys:", train_batch.keys(include_none=False))
print("train image shape:", train_batch.image.shape)
print("train label unique:", train_batch.gt_label.unique())

FileNotFoundError: Path does not exist: /home/f74134118/ADer/src/data/dcase-2020-spectrogram

In [ ]:
from anomalib.models import Padim
model = Padim(backbone="resnet18")
engine = Engine()
engine.fit(model=model, datamodule=dm)
engine.test(model=model, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configur

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor  │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator     │      0 │ train │     0 │
│ 3 │ model          │ PadimModel    │  2.8 M │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

`Trainer.fit` stopped: `max_epochs=1` reached.


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in 
true positive score
  warnings.warn(*args, **kwargs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.4336099922657013     │
│       image_F1Score       │    0.3352283835411072     │
│        pixel_AUROC        │            0.0            │
│       pixel_F1Score       │            0.0            │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 0.4336099922657013,
  'image_F1Score': 0.3352283835411072,
  'pixel_AUROC': 0.0,
  'pixel_F1Score': 0.0}]